# Projeto: Funções Open Weather Map API

### Objetivo  
Este notebook implementa funções auxiliares para coletar, extrair e estruturar dados meteorológicos de forma padronizada da Open Weather Map API. Ele permite gerar DataFrames de previsão para múltiplas cidades brasileiras, com foco em integração com pipelines de dados.

### Informações Técnicas

- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks
 
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 14/05/2025 | Gabriel Rost     |  1.0   | Criação de funções auxiliares para coleta de dados|
| 16/05/2025 | Lucas Zorn   |  1.1   | Criação de notebook|
| 19/05/2025 | Evelise Dias   |  1.1   | Inclusão de documentação|
| 21/05/2025 | Gabriel Rost   |  1.2   | Alteração de Funções|
| 25/05/2025 | Gabriel Rost   |  1.2   | Revisão de código|

## Configurações Iniciais

In [0]:
%run ./funcoes_config/

In [0]:
CHAVE_API = enum_configs.OPEN_WEATHER_MAP_KEY.value

## Função para coleta de dados na API OpenWeatherMap

### Função open_weather_map

In [0]:
def open_weather_map(lat, long):
    """
    Consulta a previsão do tempo para os próximos dias usando a API OpenWeatherMap.

    Args:
        lat (float): Latitude da localização desejada.
        long (float): Longitude da localização desejada.

    Returns:
        dict or None: Dicionário com os dados da previsão do tempo se a requisição for bem-sucedida;
                      None em caso de falha na requisição ou erro na comunicação com a API.
    """

    # Define parâmetros adicionais para a requisição
    lang = "pt_br"        # Define o idioma da resposta como português brasileiro
    unit = "metric"       # Define a unidade de medida como sistema métrico (°C, m/s)
    chave_api = CHAVE_API

    API_URL = f"https://api.openweathermap.org/data/3.0/onecall?lat={lat}&lon={long}&exclude=hourly,minutely&appid={chave_api}&units=metric"

    try:
        # Envia a requisição HTTP GET à API
        response = requests.get(API_URL)

        # Verifica se a resposta foi bem-sucedida (código 200)
        if response.status_code == 200:
            # Converte o conteúdo da resposta para JSON
            dados = response.json()
            return dados
        else:
            # Em caso de erro, imprime mensagem com o código e motivo
            print(f"Erro na requisição: {response.status_code} - {response.text}")
            return None

    # Captura exceções relacionadas a falhas na conexão ou na requisição
    except requests.exceptions.RequestException as e:
        print(f"Ocorreu um erro ao consultar a API: {e}")
        return None

### Função coletar_dados_api_open_weather_map

In [0]:

def coletar_dados_api_open_weather_map():    
    """
    Coleta dados meteorológicos diários para diversas capitais do Brasil
    utilizando a API OpenWeatherMap.

    Returns:
        list: Lista de dicionários contendo a previsão do tempo diária
              para cada capital, incluindo temperaturas, vento e precipitação.
    """
    resultado = []
    fonte = "openweathermap"

    # Itera sobre a lista de capitais definida na configuração
    for capital in enum_configs.CAPITAIS_BRASIL_WEATHER_API_MAP.value:
        cidade = capital["nome_capital"]
        estado = capital["uf"]
        lat = capital["latitude"]
        lon = capital["longitude"]

        # Consulta a API para a localização atual
        dados = open_weather_map(lat, lon)

        # Verifica se os dados diários estão disponíveis
        if "daily" not in dados:
            print(f"[ERRO] Dados diários ausentes para {cidade} ({estado})")
            continue
        
        # Processa os dados de previsão para cada dia
        for dia in dados["daily"]:
            data_previsao = datetime.datetime.utcfromtimestamp(dia["dt"]).strftime("%Y-%m-%d")
            temperatura_min = dia["temp"]["min"]
            temperatura_max = dia["temp"]["max"]
            vento_kmh = round(dia["wind_speed"] * 3.6, 2) # Conversão de m/s para km/h
            precipitacao = dia.get("rain", 0) # Pode não estar presente em todos os dias

            resultado.append({
                "data_previsao": f"{data_previsao}",
                "cidade": f"{cidade}",
                "estado": f"{estado}",
                "fonte": f"{fonte}",
                "umidade_minima_dia": f"{None}",
                "umidade_maxima_dia": f"{None}", # A API fornece apenas a média de umidade
                "temperatura_minima_dia": f"{temperatura_min}",
                "temperatura_maxima_dia": f"{temperatura_max}",
                "vel_vento_media_km": f"{vento_kmh}",
                "precipitacao": f"{precipitacao}"
            })

    return resultado